# CALCUALTE GENE-SET BURDEN SCORES 

# AML GENE EXPRESSION COX REGRESSION
This script calculates the rare variant polygenic risk score for gene sets derived from bulk RNA-seq cox regression analysis of acute myeloid leukemia. The gene-level polygenic risk score files are first downloaded from the workspace bucket. Each gene set is then processed in a loop. For each gene in a given set, the script identifies the corresponding gene-level polygenic risk score file and adds its value to a running total. The final sum represents the polygenic risk score for the entire gene set.

In [ ]:
# Load packages 
import re
import os
import gc
import glob
import pandas as pd 
from biomart import BiomartServer
from functools import reduce

In [ ]:
# Set bucket environment
my_bucket = os.getenv('WORKSPACE_BUCKET')

In [ ]:
# Remove burden polygenic risk score files 
!rm -rf *PRS.tsv

In [ ]:
# Create directory for gene lists 
!rm -rf gene_list
!mkdir -p gene_list 

In [ ]:
# Copy gene-sets from workspace bucket 
!gsutil cp {my_bucket}/data/gene_burden_analysis/gene_sets/rna_seq_cox_regression/* gene_list

In [ ]:
# Create directory for burden scores 
!mkdir -p burden_sum_scores

In [ ]:
%%bash 
# Set workspace bucket containing the gene burden sum scores 
my_bucket="gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe"
# Create directory for gene burden sum scores 
mkdir -p burden_scores 
# Loop to download file for the burden sum score for each gene 
for chrom in {1..22}; do
  mkdir -p burden_scores/burden_scores_chr${chrom}
  gsutil cp "${my_bucket}/data/gene_burden_analysis/burden_sum_scores/gene_burden_sum_scores_chr${chrom}/*.sscore" burden_scores/burden_scores_chr${chrom}
done

In [ ]:
# Print chromosome directories for burden sum scores 
!ls burden_scores

In [ ]:
# Print gene lists 
!ls gene_list

In [ ]:
# =============================================================================================================
# Calculate gene-set polygenic risk score for each gene-set
# =============================================================================================================

# Load your reference genes list first
gene_folder = "gene_list"
gene_files = sorted(glob.glob(os.path.join(gene_folder, "*")))

for gene_filename in gene_files: 
    # Load your reference genes list first
    genes = pd.read_csv(gene_filename, sep='\t')
    genes["gene_id"] = genes['gene_id'].str.split('.').str[0]
    # File name 
    match = re.search(r'/([^/]+)\.tsv$', gene_filename)
    if match:
        name = match.group(1)
        
    master = None  # Initialize before looping over chromosomes
    for chrom in range(1, 23):
        print(f"Processing chromosome {chrom} ....")

        folder_path = f"burden_scores/burden_scores_chr{chrom}"
        files = sorted(glob.glob(os.path.join(folder_path, "*")))

        if not files:
            print(f"No files found for chromosome {chrom}, skipping...")
            continue
            
         # Merge chromosome master once per chromosome
        if master is None:
            # If first chromosome, assign master as merged file 
            master = pd.read_csv(files[0], sep="\t")[["#FID", "IID"]]
            master["TOTAL_SCORE1_SUM"] = 0.0  # running sum initialized
        else:
            master = master
    
        for filename in files:
            # Find match for gene name 
            match = re.search(r"(ENSG\d+)", filename)
            if not match:
                continue
            ensg_id = match.group(1)
            if ensg_id not in genes["gene_id"].unique():
                continue
                
            # Print gene that is currently being processed 
            print(f"Processing gene {ensg_id} ....")

            df = pd.read_csv(filename, sep="\t")
            if not (df["NAMED_ALLELE_DOSAGE_SUM"] == df["SCORE1_SUM"]).all():
                continue

            # Add this gene’s score to running total
            master = master.merge(df[["#FID", "IID", "SCORE1_SUM"]],
                                  on=["#FID", "IID"], how="outer")

            master["TOTAL_SCORE1_SUM"] += master["SCORE1_SUM"]
            master.drop(columns=["SCORE1_SUM"], inplace=True)
            # Print chromosome shape 
        print(f"Chromosome {chrom} merge complete. Master shape: {master.shape}")
            
    # Print shape of merge information 
    print("All chromosomes merged.")
    print("Final shape:", master.shape)
    
    # Save the file 
    master.to_csv(f"{name}-PRS.tsv", sep ='\t', index=False)


In [ ]:
# Check the presence of the prs files 
!ls *PRS.tsv

In [ ]:
# Save prs files to workspace bucket 
!gsutil cp *PRS.tsv {my_bucket}/data/gene_burden_analysis/burden_gene_sets/rna_seq_cox_regression/ 